In [63]:
import numpy as np
import pandas as pd

In [64]:
df = pd.read_csv('/content/diabetes.csv')

In [65]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72.0,35,169.5,33.6,0.627,50,1
1,1,85,66.0,29,102.5,26.6,0.351,31,0
2,8,183,64.0,32,169.5,23.3,0.672,32,1
3,1,89,66.0,23,94.0,28.1,0.167,21,0
4,0,137,40.0,35,168.0,43.1,2.288,33,1


In [66]:
df.shape

(768, 9)

In [67]:
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.495990
BloodPressure,0.174469
SkinThickness,0.295138
Insulin,0.377081
BMI,0.315577
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [68]:
X = df.iloc[:,:-1].values
y = df.iloc[:,-1].values

In [69]:
from sklearn.preprocessing import StandardScaler
scalar = StandardScaler()
X_scaled = scalar.fit_transform(X)

In [70]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X_scaled,y,test_size=0.2,random_state =1)

In [71]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Dropout

In [72]:
from keras.src.metrics import metric
model = Sequential()
model.add(Dense(32,activation = 'relu',input_dim =8))
model.add(Dense(1,activation = 'sigmoid'))
model.compile(optimizer = 'Adam',loss ='binary_crossentropy',metrics=['accuracy'])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [73]:
model.fit(X_train,y_train,batch_size=32,epochs = 100,validation_data=(X_test,y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.5065 - loss: 0.7098 - val_accuracy: 0.6104 - val_loss: 0.6582
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6221 - loss: 0.6388 - val_accuracy: 0.6883 - val_loss: 0.5972
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6580 - loss: 0.5886 - val_accuracy: 0.7727 - val_loss: 0.5505
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6987 - loss: 0.5521 - val_accuracy: 0.7597 - val_loss: 0.5166
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7296 - loss: 0.5253 - val_accuracy: 0.7727 - val_loss: 0.4909
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7362 - loss: 0.5054 - val_accuracy: 0.7857 - val_loss: 0.4702
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7459 - loss: 0.4904 - val_accuracy: 0.7922 - val_loss: 0.4548
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7573 - loss: 0.4777 - val_accuracy: 0.7987 

In [74]:
pip install -U keras-tuner

In [75]:
import keras_tuner as kt

In [96]:
def build_model(hp):
  model = Sequential()
  count =0
  for i in range(hp.Int('num_layers',min_value=1,max_value=8)):
    if count == 0:
      model.add(Dense(hp.Int('units' + str(i),min_value=32,max_value =128,step=8),
                      activation = hp.Choice('activation' + str(i),values = ['relu','tanh']),input_dim = 8))
      model.add(Dropout(hp.Choice('dropout'+str(i),values = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))
    else:
      model.add(Dense(hp.Int('units' + str(i),min_value=32,max_value =128,step=8),
                      activation = hp.Choice('activation' + str(i),values = ['relu','tanh'])))
      model.add(Dropout(hp.Choice('dropout'+str(i),values = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))
    count += 1

  model.add(Dense(1,activation ='sigmoid'))
  model.compile(optimizer=hp.Choice('optimizer',values=['rmsprop','adam','sgd','adadelta','nadam']),loss= 'binary_crossentropy',metrics=['accuracy'])
  return model

In [97]:
tuner = kt.RandomSearch(build_model,objective='val_accuracy',max_trials=3,directory='mydir',project_name='_rekon')

In [98]:
tuner.search(X_train,y_train,epochs = 5,validation_data = (X_test,y_test))

Trial 3 Complete [00h 00m 04s]
val_accuracy: 0.6428571343421936

Best val_accuracy So Far: 0.7922077775001526
Total elapsed time: 00h 00m 16s


In [100]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 3,
 'units0': 32,
 'activation0': 'relu',
 'dropout0': 0.1,
 'optimizer': 'sgd',
 'units1': 32,
 'activation1': 'tanh',
 'dropout1': 0.1,
 'units2': 56,
 'activation2': 'relu',
 'dropout2': 0.4,
 'units3': 80,
 'activation3': 'tanh',
 'dropout3': 0.1}

In [101]:
model = tuner.get_best_models(num_models=1)[0]

In [102]:
model.fit(X_train,y_train,epochs=200,initial_epoch =6,validation_data=(X_test,y_test))

Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6824 - loss: 0.6101 - val_accuracy: 0.7857 - val_loss: 0.5763
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7085 - loss: 0.5919 - val_accuracy: 0.7792 - val_loss: 0.5595
Epoch 9/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7036 - loss: 0.5877 - val_accuracy: 0.7727 - val_loss: 0.5453
Epoch 10/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7117 - loss: 0.5687 - val_accuracy: 0.7727 - val_loss: 0.5331
Epoch 11/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7215 - loss: 0.5495 - val_accuracy: 0.7727 - val_loss: 0.5218
Epoch 12/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7427 - loss: 0.5426 - val_accuracy: 0.7792 - val_loss: 0.5110
Epoch 13/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7280 - loss: 0.5464 - val_accuracy: 0.7857 - val_loss: 0.5033
Epoch 14/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7313 - loss: 0.5333 - val_accuracy: 0.79